## Top-Performing Locations
### Goal: 
Identify best and worst-performing store locations.
### Why it matters: 
Informs decisions about promotions, staffing, or expansion.
### How to do it:
- Group order_items by location_id (or store_id if available)
- Calculate:
- • Total revenue
- • Average order value
- • Orders per day/week
- Rank locations based on revenue

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')

In [0]:
df_restaurant_rev = (df_fact_order.groupBy('restaurant_id')
        .agg(
            F.round(F.sum('item_price'),2).alias('total_revenue'),
            F.round(F.avg('item_price'),2).alias('avg_order_revenue'))
        .orderBy(F.col('total_revenue').desc())
)

In [0]:
df_restaurant_rev.display()

In [0]:
df_restaurant_avg_ord = df_fact_order.groupBy('restaurant_id','order_id').agg(F.avg('item_price').alias('avg_order_revenue')).orderBy(F.col('avg_order_revenue').desc())

In [0]:
window = Window.orderBy(F.col('total_revenue').desc())
df_restaurant_rev_rnk = df_restaurant_rev.withColumn('rnk',
                                F.rank().over(window))
df_restaurant_rev_rnk.display()

### Orders per day

In [0]:
df_order_daywise = (df_fact_order.groupBy('restaurant_id','date_key')
        .agg(
            F.count_distinct('order_id').alias('no_of_orders'))
        .orderBy('restaurant_id',F.to_date(F.col('date_key'),'dd-MM-yyyy'))
)
df_order_daywise.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.restaurant_performance;
DROP TABLE IF EXISTS global_partner_project.mart.restaurant_daywise_orders;

In [0]:
df_restaurant_rev_rnk.write.mode('append').saveAsTable('global_partner_project.mart.restaurant_performance')
df_order_daywise.write.mode('append').saveAsTable('global_partner_project.mart.restaurant_daywise_orders')